Projeto Default of Credit Card Clients

DATASET: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

PROBLEMA
Cliente: empresa de cartão de crédito

Objetivo: desenvolver um modelo que faça a predição se uma conta ficara inadimplete no próximo mês

Recebemos um dataset que inclui dados demográficos e financeiros recentes (últimos seis meses) com 30.000 contas de titulares. Históricos de contas que no final de seis meses.

TARGET: se proprietario da conta ficou inadimplente, ou seja, não fez o pagamento minimo.

In [0]:

"""
DESCRIÇÃO DO DATASET:

X1: Amount of the given credit (NT dollar): it includes both the individual consumer credit and his/her family (supplementary) credit. 
X2: Gender (1 = male; 2 = female). 
X3: Education (1 = graduate school; 2 = university; 3 = high school; 4 = others). 
X4: Marital status (1 = married; 2 = single; 3 = others). 
X5: Age (year). 
X6 - X11: History of past payment. We tracked the past monthly payment records (from April to September, 2005) as follows: X6 = the repayment status in September, 2005; 
X7 = the repayment status in August, 2005; . . .;
X11 = the repayment status in April, 2005. The measurement scale for the repayment status is: -1 = pay duly; 1 = payment delay for one month; 2 = payment delay for two months; . . .; 8 = payment delay for eight months; 9 = payment delay for nine months and above.
X12-X17: Amount of bill statement (NT dollar). X12 = amount of bill statement in September, 2005; X13 = amount of bill statement in August, 2005; . . .; X17 = amount of bill statement in April, 2005. 
X18-X23: Amount of previous payment (NT dollar). X18 = amount paid in September, 2005; X19 = amount paid in August, 2005; . . .;X23 = amount paid in April, 2005.

default payment next month: 0 = the client will pay next month; 1 = the client will default payment in the next
"""

In [0]:
%pip install xgboost

In [0]:
#Importando bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, when
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression


In [0]:
df = spark.table("workspace.default.default_of_credit_card_clients")

In [0]:
df.display()

### EXPLORATÓRIA 

FEATURES CATEGORICAS

In [0]:
#Frequencia das features categoricas
df.groupBy('SEX').count().display()


In [0]:
df.groupBy('EDUCATION').count().orderBy("EDUCATION", ascending=True).display()
#Valores 0, 5, 6 não tem uma descrição, e o cliente desconhece essa categoria: Solução agrupar em outros, não perde a informação

In [0]:
df.groupBy('MARRIAGE').count().display()
#o valor 0 não tem uma descrição na documentação, incluir na categoria outros

In [0]:
df.groupBy('PAY_0').count().display()


In [0]:
df.groupBy('PAY_2').count().display()

In [0]:
"""
Registro de pagamentos passados -1, 1, 2, 3, 4, 5, 6, 7, 8 e 9 

os 0 e -2 não estão na descrição da documentação. O cliente informou:

-2: significa que a conta começou o mês sem valor a ser pago e o credito não foi usado

0: significa que o pagamento mínimo foi feito, mas o saldo total devedor não foi pago
"""

In [0]:
df.groupBy('default payment next month').count().display()

FEATURES NUMERICAS

In [0]:
df[['LIMIT_BAL', 'AGE']].describe().display()

In [0]:
#Tranforma em dataframe Pandas
df_pandas = df.toPandas()

In [0]:
#Grafico de distribuição
df_pandas[['LIMIT_BAL', 'AGE']].hist()

In [0]:
#Valor da fatura dos últimos seis meses
df_pandas[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3','BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].describe()


In [0]:
#Valore de pagamento anterior
df_pandas[['PAY_AMT1','PAY_AMT2','PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5','PAY_AMT6']].describe()

In [0]:
#Checar valores nulos
df_pandas.isnull().sum()

### SEPARAR TREINO E TESTE

In [0]:
from sklearn.model_selection import train_test_split

In [0]:
#Separar as bases de treino e teste
X = df_pandas.drop('default payment next month', axis = 1)
y = df_pandas['default payment next month']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify =y)

In [0]:
df_train = X_train.copy()

In [0]:
df_train['EDUCATION'] = df_train['EDUCATION'].replace(to_replace=([0, 5, 6]), value=4)

In [0]:
df_train['EDUCATION'].value_counts()

In [0]:
#Valor 0 incluir na categoria 3
df_train['MARRIAGE'] = df_train['MARRIAGE'].replace(0, 3)

In [0]:
df_train['MARRIAGE'].value_counts()

In [0]:
#CODIFICAÇÃO ONE HOT(one hot enconding)
#Engenharia reversa: transformados os rotulos numericos em strings de acodo com a descrição do documento
#Criar uma coluna EDUCATION_CAT

df_train['EDUCATION_CAT'] = 'NONE'

cat_mapping = {
1: 'graduate school',
2: 'university',
3: 'high school',
4: 'edu_others'
}

#Função map, para preencher mapeando deacordo com os valores
df_train['EDUCATION_CAT'] = df_train['EDUCATION'].map(cat_mapping)

#Cada categoria da coluna EDUCATION_CAT vira uma categoria
edu_ohe = pd.get_dummies(df_train['EDUCATION_CAT'], dtype=int)

#Concatenar
df_ohe = pd.concat([df_train, edu_ohe], axis=1)
df_ohe[['EDUCATION_CAT', 'graduate school', 'high school', 'edu_others', 'university' ]]

In [0]:
df_ohe["SEX"] = df_ohe["SEX"].replace({1: 0, 2: 1})

In [0]:
#one hot encoding da variável MARRIAGE
df_ohe['MARRIAGE_CAT'] = 'NONE'

cat_mapping = {
1: 'married',
2: 'single',
3: 'marriage_others'
}

df_ohe['MARRIAGE_CAT'] = df_ohe['MARRIAGE'].map(cat_mapping)

#Cada categoria da coluna EDUCATION_CAT vira uma categoria
marr_ohe = pd.get_dummies(df_ohe['MARRIAGE_CAT'], dtype=int)


df_marr = pd.concat([df_ohe, marr_ohe], axis=1)
df_marr[['MARRIAGE_CAT', 'married','single', 'marriage_others']]

In [0]:
df_treino = df_marr[['LIMIT_BAL',  'AGE', 'BILL_AMT1', 'BILL_AMT2','BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6','graduate school','university','high school', 'edu_others', 'married' ,'single', 'marriage_others']]

### RELAÇÃO COM A TARGET

In [0]:
#Base treino
df_treino_cp = df_marr.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

In [0]:
#Concatenar as duas tabelas pela linha

df_analise = pd.concat([df_treino_cp, y_train], axis = 1)

In [0]:
df_analise.display()

In [0]:
df_analise['default payment next month'].value_counts()

In [0]:
plt.figure(figsize = (8, 6))
sns.countplot(x = 'default payment next month', data = df_analise)

In [0]:
#Target vs sexo: 0 = masculino e 1: feminino
df_analise['SEX'].value_counts()

In [0]:
df_categorias = df_analise[['SEX', 'EDUCATION', 'MARRIAGE']]

for col in df_categorias:
    plt.figure(figsize=(8,6))
    fig, axes = plt.subplots(ncols=2, figsize= (13,8))
    df_analise[col].value_counts().plot(kind='pie', ax = axes[0], subplots = True)
    sns.countplot(x=col, hue = 'default payment next month', data = df_analise)
    plt.show()



In [0]:
df_test = X_test.copy()

In [0]:
df_test['EDUCATION'] = df_test['EDUCATION'].replace(to_replace=([0, 5, 6]), value=4)

In [0]:
#Valor 0 incluir na categoria 3
df_test['MARRIAGE'] = df_test['MARRIAGE'].replace(0, 3)

In [0]:
#CODIFICAÇÃO ONE HOT(one hot enconding)
df_test['EDUCATION_CAT'] = 'NONE'

cat_mapping = {
1: 'graduate school',
2: 'university',
3: 'high school',
4: 'edu_others'
}

#Função map, para preencher mapeando deacordo com os valores
df_test['EDUCATION_CAT'] = df_test['EDUCATION'].map(cat_mapping)

#Cada categoria da coluna EDUCATION_CAT vira uma categoria
edu_ohe = pd.get_dummies(df_test['EDUCATION_CAT'], dtype=int)

#Concatenar
df_ohe_test = pd.concat([df_test, edu_ohe], axis=1)
df_ohe_test[['EDUCATION_CAT', 'graduate school', 'high school', 'edu_others', 'university' ]]

In [0]:
df_ohe_test["SEX"] = df_ohe_test["SEX"].replace({1: 0, 2: 1})

In [0]:
df_ohe_test['MARRIAGE_CAT'] = 'NONE'

cat_mapping = {
1: 'married',
2: 'single',
3: 'marriage_others'
}

df_ohe_test['MARRIAGE_CAT'] = df_ohe_test['MARRIAGE'].map(cat_mapping)

#Cada categoria da coluna EDUCATION_CAT vira uma categoria
marr_ohe_test= pd.get_dummies(df_ohe_test['MARRIAGE_CAT'], dtype=int)


df_marr_test = pd.concat([df_ohe_test, marr_ohe_test], axis=1)
df_marr_test[['MARRIAGE_CAT', 'married','single', 'marriage_others']]

In [0]:
df_teste = df_marr_test[['LIMIT_BAL',  'AGE', 'BILL_AMT1', 'BILL_AMT2','BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6','graduate school','university','high school', 'edu_others', 'married' ,'single', 'marriage_others']]

### Modelo Regressão Logistica

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay,
                            RocCurveDisplay)

#Definir experimento
experiment_name = "/Users/ve_amparo@yahoo.com.br/modelo_inadimplencia"
mlflow.set_experiment(experiment_name)

#Base treino
df_treino = df_treino.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

#Base teste
df_teste = df_teste.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

#Iniciar Run
with mlflow.start_run(run_name='Regressao logistica'):

    
    modelo_rg = LogisticRegression(
                max_iter=1000,
                class_weight='balanced')

    #1.Treinando o modelo
    modelo_rg.fit(df_treino, y_train)

    #2.Predição
    y_pred = modelo_rg.predict(df_teste)
    y_prob = modelo_rg.predict_proba(df_teste)[:, 1]

    #3.Metricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    #Log metricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric('auc', auc)

    #4.Matriz confusão
    cm = confusion_matrix(y_test, y_pred)

    fig_cm, ax = plt.subplots()
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues')
    plt.title('Matriz de Confusão')

    mlflow.log_figure(fig_cm, "confusion_matriz.png")
    plt.close(fig_cm)

    #5.Curva ROC
    fig_roc, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
    plt.title("Curva ROC")
   

    mlflow.log_figure(fig_roc, "roc_curve.png")
    plt.close(fig_roc)

    #6.Feature importance
    feature_importance = pd.DataFrame({
                    'feature': df_treino.columns, 
                    'importance': modelo_rg.coef_[0]
                    }).sort_values(by='importance', ascending=False)

    fig_imp, ax = plt.subplots()
    sns.barplot(x='importance', y='feature', data=feature_importance, ax=ax)
    plt.title('Feature Importance')

    mlflow.log_figure(fig_imp, "feature_importance.png")
    plt.close(fig_imp)


    #7.Log Modelo
    mlflow.sklearn.log_model(modelo_rg, "modelo_logistico")


### Modelo RandomForestClassifier

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay,
                            RocCurveDisplay)

#Definir experimento
experiment_name = "/Users/ve_amparo@yahoo.com.br/modelo_inadimplencia"
mlflow.set_experiment(experiment_name)

#Base treino
df_treino = df_treino.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

#Base teste
df_teste = df_teste.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

#Iniciar Run
with mlflow.start_run(run_name='Random Forest'):

    modelo_rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
    )

    #1.Treinando o modelo
    modelo_rf.fit(df_treino, y_train)

    #2.Predição
    y_pred = modelo_rf.predict(df_teste)
    y_prob = modelo_rf.predict_proba(df_teste)[:, 1]

    #3.Metricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    #Log metricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric('auc', auc)

    #4.Matriz confusão
    cm = confusion_matrix(y_test, y_pred)

    fig_cm, ax = plt.subplots()
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues')
    plt.title('Matriz de Confusão')

    mlflow.log_figure(fig_cm, "confusion_matriz.png")
    plt.close(fig_cm)

    #5.Curva ROC
    fig_roc, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
    plt.title("Curva ROC")
   
    ax.plot([0, 1], [0, 1], linestyle='--')
    mlflow.log_figure(fig_roc, "roc_curve.png")
    plt.close(fig_roc)

    #6.Feature importance
    feature_importance = pd.DataFrame({
                    'feature': df_treino.columns, 
                    'importance': modelo_rf.feature_importances_
                    }).sort_values(by='importance', ascending=False)

    fig_imp, ax = plt.subplots()
    sns.barplot(x='importance', y='feature', data=feature_importance, ax=ax)
    plt.title('Feature Importance')

    mlflow.log_figure(fig_imp, "feature_importance.png")
    plt.close(fig_imp)


    #7.Log Modelo
    mlflow.sklearn.log_model(modelo_rf, "modelo_Random_Forest")


### Modelo XGBoost

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay,
                            RocCurveDisplay)

#Definir experimento
experiment_name = "/Users/ve_amparo@yahoo.com.br/modelo_inadimplencia"
mlflow.set_experiment(experiment_name)

#Base treino
df_treino = df_treino.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

#Base teste
df_teste = df_teste.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

#Iniciar Run
with mlflow.start_run(run_name='XGBoost'):

    modelo_xgb = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1,  # ajustar se desbalanceado
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )

    #1.Treinando o modelo
    modelo_xgb.fit(df_treino, y_train)

    #2.Predição
    y_pred = modelo_xgb.predict(df_teste)
    y_prob = modelo_xgb.predict_proba(df_teste)[:, 1]

    #3.Metricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    #Log metricas
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric('auc', auc)

    #4.Matriz confusão
    cm = confusion_matrix(y_test, y_pred)

    fig_cm, ax = plt.subplots()
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues')
    plt.title('Matriz de Confusão')

    mlflow.log_figure(fig_cm, "confusion_matriz.png")
    plt.close(fig_cm)

    #5.Curva ROC
    fig_roc, ax = plt.subplots()
    RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
    plt.title("Curva ROC")
   
    ax.plot([0, 1], [0, 1], linestyle='--')
    mlflow.log_figure(fig_roc, "roc_curve.png")
    plt.close(fig_roc)

    #6.Feature importance
    feature_importance = pd.DataFrame({
                    'feature': df_treino.columns, 
                    'importance': modelo_xgb.feature_importances_
                    }).sort_values(by='importance', ascending=False)

    fig_imp, ax = plt.subplots()
    sns.barplot(x='importance', y='feature', data=feature_importance, ax=ax)
    plt.title('Feature Importance')

    mlflow.log_figure(fig_imp, "feature_importance.png")
    plt.close(fig_imp)


    #7.Log Modelo
    mlflow.sklearn.log_model(modelo_xgb, "modelo_XGBoost")


### 

In [0]:
#Sexo com target, sexo masculino tem maior taxa de default (inadimplencia)
X.groupby('SEX').agg({'default payment next month': 'mean'}).plot.bar(legend=False)
plt.ylabel('Taxa')
plt.xlabel('Genero')

In [0]:
#Em relação ao grau de instrução: as categorias 2 e 3 são que tem maior risco de inandimplência
#Indica que é uma variável potencialmente preditiva para modelo
df_pandas.groupby("EDUCATION").agg({'default payment next month': 'mean'}).plot.bar(legend = True)
plt.ylabel('Taxa')
plt.xlabel('Grau de instrução')

In [0]:

resumo = df_pandas.groupby("EDUCATION").agg(
          volume = ('default payment next month', 'count'),
          taxa = ('default payment next month', 'mean')
).sort_values(by = 'taxa', ascending = False)

resumo.display()


In [0]:
#Em relação a idade: as medias da idades nos dois grupos é muito próximo, não indica que é uma variável potencialmente preditiva para o modelo
df_pandas.groupby('default payment next month')['AGE'].mean()